In [0]:
from pyspark.sql.functions import col, upper, current_timestamp

df_po_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/raw_data/purchase_orders_raw.csv")
)

In [0]:
df_po_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.pharma_bronze.bronze_purchase_orders")

In [0]:
df_po_silver = (
    df_po_bronze
    .withColumn(
        "po_status",
        upper(col("po_status"))
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)

In [0]:
df_po_silver.groupBy("po_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
from delta.tables import DeltaTable

delta_target = DeltaTable.forName(
    spark,
    "workspace.pharma_silver.silver_purchase_orders"
)

(
    delta_target.alias("target")
    .merge(
        df_po_silver.alias("source"),
        "target.po_id = source.po_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
df_po_final = spark.table(
    "workspace.pharma_silver.silver_purchase_orders"
)

print("Total Silver PO records:", df_po_final.count())

print("Duplicate PO IDs:")

df_po_final.groupBy("po_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
from pyspark.sql.functions import lit

new_real_po = df_po_silver.limit(1).withColumn(
    "po_id",
    lit(999999998)
)

display(new_real_po)

In [0]:
(
    delta_target.alias("target")
    .merge(
        new_real_po.alias("source"),
        "target.po_id = source.po_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
df_po_final = spark.table(
    "workspace.pharma_silver.silver_purchase_orders"
)

print("Total Silver PO records:", df_po_final.count())

In [0]:
updated_real_po = new_real_po.withColumn(
    "quantity_ordered",
    col("quantity_ordered") + 100
)

display(updated_real_po)

In [0]:
(
    delta_target.alias("target")
    .merge(
        updated_real_po.alias("source"),
        "target.po_id = source.po_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
df_po_final = spark.table(
    "workspace.pharma_silver.silver_purchase_orders"
)

test_po = df_po_final.filter(
    col("po_id") == 999999998
)

print("Total Silver PO records:", df_po_final.count())
print("Test PO count:", test_po.count())

display(test_po)

In [0]:
from delta.tables import DeltaTable

delta_target = DeltaTable.forName(
    spark,
    "workspace.pharma_silver.silver_purchase_orders"
)

delta_target.delete(
    "po_id = 999999998"
)

In [0]:
df_po_final = spark.table(
    "workspace.pharma_silver.silver_purchase_orders"
)

print("Final Silver PO records:", df_po_final.count())

display(
    df_po_final.filter(
        col("po_id") == 999999998
    )
)

In [0]:
df_po_bronze_current = spark.table(
    "workspace.pharma_bronze.bronze_purchase_orders"
)

df_po_silver_current = spark.table(
    "workspace.pharma_silver.silver_purchase_orders"
)

print("Bronze records:", df_po_bronze_current.count())
print("Silver records:", df_po_silver_current.count())

In [0]:
new_pos = df_po_bronze_current.join(
    df_po_silver_current.select("po_id"),
    on="po_id",
    how="left_anti"
)

print("New POs:", new_pos.count())

display(new_pos)

In [0]:
simulated_new_po = df_po_bronze_current.limit(1).withColumn(
    "po_id",
    lit(999999997)
)

display(simulated_new_po)

In [0]:
df_po_bronze_incremental = df_po_bronze_current.unionByName(
    simulated_new_po
)

print(
    "Simulated Bronze records:",
    df_po_bronze_incremental.count()
)

In [0]:
new_pos = df_po_bronze_incremental.join(
    df_po_silver_current.select("po_id"),
    on="po_id",
    how="left_anti"
)

print("New POs:", new_pos.count())

display(new_pos)

In [0]:
new_pos_silver = (
    new_pos
    .withColumn(
        "po_status",
        upper(col("po_status"))
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)

display(new_pos_silver)

In [0]:
(
    delta_target.alias("target")
    .merge(
        new_pos_silver.alias("source"),
        "target.po_id = source.po_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
df_po_silver_final = spark.table(
    "workspace.pharma_silver.silver_purchase_orders"
)

print("Final Silver PO records:", df_po_silver_final.count())

print("New PO count:", 
      df_po_silver_final
      .filter(col("po_id") == 999999997)
      .count())

display(
    df_po_silver_final
    .filter(col("po_id") == 999999997)
)

In [0]:
delta_target.delete(
    "po_id = 999999997"
)

In [0]:
df_po_silver_final = spark.table(
    "workspace.pharma_silver.silver_purchase_orders"
)

print("Final Silver PO records:", df_po_silver_final.count())

display(
    df_po_silver_final.filter(
        col("po_id") == 999999997
    )
)

In [0]:
null_po_ids = df_po_silver.filter(
    col("po_id").isNull()
).count()

print("NULL PO IDs:", null_po_ids)

In [0]:
duplicate_po_ids = (
    df_po_silver
    .groupBy("po_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate PO IDs:", duplicate_po_ids)

In [0]:
negative_quantity = df_po_silver.filter(
    col("quantity_ordered") < 0
).count()

print("Negative quantities:", negative_quantity)

In [0]:
df_po_silver.select("po_status").distinct().show()

In [0]:
valid_statuses = [
    "RECEIVED",
    "PENDING",
    "CANCELLED",
    "APPROVED"
]

invalid_status_count = df_po_silver.filter(
    ~col("po_status").isin(valid_statuses)
).count()

print("Invalid PO statuses:", invalid_status_count)

In [0]:
df_po_bronze.count()

In [0]:
df_po_bronze.printSchema()

In [0]:
df_po_bronze.show(1, truncate=False)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, DateType, StringType
from datetime import date

new_po = spark.createDataFrame(
    [
        (21, 5, 1, date(2026, 8, 20), 600, 250.00, date(2026, 9, 5), "Approved")
    ],
    schema=StructType([
        StructField("po_id", IntegerType(), True),
        StructField("supplier_id", IntegerType(), True),
        StructField("drug_id", IntegerType(), True),
        StructField("order_date", DateType(), True),
        StructField("quantity_ordered", IntegerType(), True),
        StructField("unit_price", DoubleType(), True),
        StructField("expected_delivery_date", DateType(), True),
        StructField("po_status", StringType(), True)
    ])
)

display(new_po)

In [0]:
new_po.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.pharma_bronze.bronze_purchase_orders")

In [0]:
spark.table(
    "workspace.pharma_bronze.bronze_purchase_orders"
).count()

In [0]:
spark.table(
    "workspace.pharma_silver.silver_purchase_orders"
).count()

In [0]:
spark.table(
    "workspace.pharma_silver.silver_purchase_orders"
).count()

In [0]:
raw_path = "/Volumes/workspace/default/raw_data/purchase_orders_raw.csv"

# Read the existing CSV content
existing_data = dbutils.fs.head(raw_path, 1000000)

# Add PO 21
new_row = "21,5,1,2026-08-20,600,250.0,2026-09-05,Approved\n"

# Write the original data + new row back
dbutils.fs.put(
    raw_path,
    existing_data + new_row,
    True
)

print("PO 21 added to raw CSV")

In [0]:
test_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/default/raw_data/purchase_orders_raw.csv")

print("Raw PO records:", test_raw.count())

In [0]:
df_po_silver_final = spark.table(
    "workspace.pharma_silver.silver_purchase_orders"
)

print("Silver PO records:", df_po_silver_final.count())

display(
    df_po_silver_final.filter(
        col("po_id") == 21
    )
)

In [0]:
from pyspark.sql.functions import col, lit, when

raw_path = "/Volumes/workspace/default/raw_data/purchase_orders_raw.csv"

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(raw_path)

df_raw_updated = df_raw.withColumn(
    "quantity_ordered",
    when(
        col("po_id") == 21,
        lit(750)
    ).otherwise(col("quantity_ordered"))
)

dbutils.fs.rm(raw_path, True)

temp_path = "/Volumes/workspace/default/raw_data/purchase_orders_temp"

df_raw_updated.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(temp_path)

files = dbutils.fs.ls(temp_path)

csv_file = [
    f.path for f in files
    if f.name.endswith(".csv")
][0]

dbutils.fs.cp(csv_file, raw_path)

dbutils.fs.rm(temp_path, True)

print("PO 21 quantity updated to 750")